# PySpark ETL + Deep Learning

This notebook uses:

- **PySpark** for preprocessing / distributed ETL
- **PyTorch** for CNN
- **TensorFlow** for LSTM
- **Hugging Face Transformers** for BERT fine-tuning
- **PEFT LoRA** for lightweight LLM fine-tuning

## Workflow

1. Create  text data
2. Run PySpark ETL and export Parquet
3. Train a TensorFlow LSTM classifier
4. Train a PyTorch CNN on synthetic image data
5. Fine-tune a tiny BERT model for text classification
6. Fine-tune a tiny GPT-2 model with LoRA

In [1]:
!pip -q install pyspark pandas pyarrow torch torchvision tensorflow transformers datasets evaluate scikit-learn accelerate peft trl sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 13.9 MB/s eta 0:00:00


In [2]:
import os
import random
from pathlib import Path
import pandas as pd
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

BASE_DIR = Path("/content/pyspark_deep_learning_demo")
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed" / "text_reviews"
ARTIFACTS_DIR = BASE_DIR / "artifacts"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

print("Base directory:", BASE_DIR)

Base directory: /content/pyspark_deep_learning_demo


In [3]:
sample_rows = [
    ("I love this product and would buy it again", 1),
    ("This is the worst service I have ever used", 0),
    ("Fast shipping and excellent quality", 1),
    ("The app crashes every time I open it", 0),
    ("Pretty good overall with only minor issues", 1),
    ("Terrible battery life and poor build quality", 0),
    ("Amazing support team and very helpful answers", 1),
    ("The screen freezes and the device overheats", 0),
    ("The interface is smooth and very intuitive", 1),
    ("Customer support never solved my problem", 0),
    ("I am impressed with the speed and accuracy", 1),
    ("This update made everything worse", 0),
    ("Absolutely fantastic user experience", 1),
    ("I regret buying this item", 0),
    ("The response time is fast and reliable", 1),
    ("Frequent bugs make it unusable", 0),
    ("It works exactly as expected", 1),
    ("The instructions are confusing and incomplete", 0),
    ("Great value for the money", 1),
    ("The product stopped working after one day", 0),
]

raw_df = pd.DataFrame(sample_rows, columns=["text", "label"])
raw_csv_path = RAW_DIR / "sample_reviews.csv"
raw_df.to_csv(raw_csv_path, index=False)

print(raw_csv_path)
raw_df.head(10)

/content/pyspark_deep_learning_demo/data/raw/sample_reviews.csv


,text,label
0,I love this product and would buy it again,1
1,This is the worst service I have ever used,0
2,Fast shipping and excellent quality,1
3,The app crashes every time I open it,0
4,Pretty good overall with only minor issues,1
5,Terrible battery life and poor build quality,0
6,Amazing support team and very helpful answers,1
7,The screen freezes and the device overheats,0
8,The interface is smooth and very intuitive,1
9,Customer support never solved my problem,0


In [4]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

spark = (
    SparkSession.builder
    .appName("PySparkTextETLColab")
    .getOrCreate()
)

spark_df = spark.read.option("header", True).csv(str(raw_csv_path))

cleaned = (
    spark_df.select("text", "label")
    .dropna(subset=["text", "label"])
    .withColumn("text", F.trim(F.col("text")))
    .filter(F.length("text") > 0)
    .withColumn("text", F.lower("text"))
    .withColumn("text", F.regexp_replace("text", r"[^a-z0-9\s]", " "))
    .withColumn("text", F.regexp_replace("text", r"\s+", " "))
    .withColumn("label", F.col("label").cast(IntegerType()))
    .dropDuplicates(["text", "label"])
)

train_df, valid_df, test_df = cleaned.randomSplit([0.7, 0.15, 0.15], seed=SEED)

train_path = PROCESSED_DIR / "train.parquet"
valid_path = PROCESSED_DIR / "valid.parquet"
test_path = PROCESSED_DIR / "test.parquet"

train_df.write.mode("overwrite").parquet(str(train_path))
valid_df.write.mode("overwrite").parquet(str(valid_path))
test_df.write.mode("overwrite").parquet(str(test_path))

print("Train rows:", train_df.count())
print("Valid rows:", valid_df.count())
print("Test rows:", test_df.count())
cleaned.show(truncate=False)

Train rows: 12
Valid rows: 6
Test rows: 2
+---------------------------------------------+-----+
|text                                         |label|
+---------------------------------------------+-----+
|the response time is fast and reliable       |1    |
|it works exactly as expected                 |1    |
|the product stopped working after one day    |0    |
|frequent bugs make it unusable               |0    |
|fast shipping and excellent quality          |1    |
|terrible battery life and poor build quality |0    |
|this update made everything worse            |0    |
|i am impressed with the speed and accuracy   |1    |
|the app crashes every time i open it         |0    |
|absolutely fantastic user experience         |1    |
|amazing support team and very helpful answers|1    |
|i regret buying this item                    |0    |
|the interface is smooth and very intuitive   |1    |
|this is the worst service i have ever used   |0    |
|pretty good overall with only minor iss

In [5]:
train_pd = pd.read_parquet(train_path)
valid_pd = pd.read_parquet(valid_path)
test_pd = pd.read_parquet(test_path)

print("Train shape:", train_pd.shape)
print("Valid shape:", valid_pd.shape)
print("Test shape:", test_pd.shape)

train_pd.head()

Train shape: (12, 2)
Valid shape: (6, 2)
Test shape: (2, 2)


,text,label
0,absolutely fantastic user experience,1
1,amazing support team and very helpful answers,1
2,fast shipping and excellent quality,1
3,frequent bugs make it unusable,0
4,great value for the money,1


In [6]:
import tensorflow as tf

MAX_TOKENS = 5000
SEQ_LEN = 64

vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=MAX_TOKENS,
    output_mode="int",
    output_sequence_length=SEQ_LEN,
)
vectorizer.adapt(train_pd["text"].astype(str).tolist())

x_train = vectorizer(tf.constant(train_pd["text"].astype(str).tolist()))
y_train = tf.constant(train_pd["label"].astype("float32").tolist())
x_valid = vectorizer(tf.constant(valid_pd["text"].astype(str).tolist()))
y_valid = tf.constant(valid_pd["label"].astype("float32").tolist())

lstm_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(SEQ_LEN,)),
    tf.keras.layers.Embedding(MAX_TOKENS, 64),
    tf.keras.layers.LSTM(64),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])

lstm_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

history = lstm_model.fit(
    x_train,
    y_train,
    validation_data=(x_valid, y_valid),
    epochs=3,
    batch_size=4,
    verbose=1,
)

lstm_out = ARTIFACTS_DIR / "lstm_text_classifier.keras"
lstm_model.save(lstm_out)
print("Saved:", lstm_out)

Epoch 1/3
3/3 ━━━━━━━━━━━━━━━━━━━━ 7s 481ms/step - accuracy: 0.4167 - loss: 0.6993 - val_accuracy: 0.6667 - val_loss: 0.6893
Epoch 2/3
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 107ms/step - accuracy: 0.5833 - loss: 0.6926 - val_accuracy: 0.3333 - val_loss: 0.6976
Epoch 3/3
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 125ms/step - accuracy: 0.5833 - loss: 0.6913 - val_accuracy: 0.3333 - val_loss: 0.7074
Saved: /content/pyspark_deep_learning_demo/artifacts/lstm_text_classifier.keras


In [7]:
sample_texts = tf.constant([
    "excellent quality and amazing support",
    "terrible bugs and awful experience",
    "it is okay not the best but usable"
])

sample_vectors = vectorizer(sample_texts)
sample_preds = lstm_model.predict(sample_vectors)

for text, pred in zip(sample_texts.numpy(), sample_preds.flatten()):
    print(text.decode("utf-8"), "->", float(pred))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 405ms/step
excellent quality and amazing support -> 0.5201404690742493
terrible bugs and awful experience -> 0.5201404690742493
it is okay not the best but usable -> 0.5201404690742493


In [8]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

class SimpleCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 8 * 8, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

transform = transforms.Compose([transforms.ToTensor()])
cnn_dataset = datasets.FakeData(
    size=512,
    image_size=(3, 32, 32),
    num_classes=2,
    transform=transform
)
cnn_loader = DataLoader(cnn_dataset, batch_size=32, shuffle=True)

cnn_model = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(cnn_model.parameters(), lr=1e-3)

for epoch in range(2):
    cnn_model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    for x, y in cnn_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = cnn_model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)

    print(f"Epoch {epoch + 1}: loss={total_loss / total:.4f}, acc={correct / total:.4f}")

cnn_out = ARTIFACTS_DIR / "cnn_model.pt"
torch.save(cnn_model.state_dict(), cnn_out)
print("Saved:", cnn_out)

Device: cpu
Epoch 1: loss=0.7034, acc=0.4570
Epoch 2: loss=0.6939, acc=0.4941
Saved: /content/pyspark_deep_learning_demo/artifacts/cnn_model.pt


In [9]:
import evaluate
from datasets import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

bert_model_name = "prajjwal1/bert-tiny"

train_ds = Dataset.from_pandas(train_pd[["text", "label"]], preserve_index=False)
valid_ds = Dataset.from_pandas(valid_pd[["text", "label"]], preserve_index=False)

bert_tokenizer = AutoTokenizer.from_pretrained(bert_model_name)

def tokenize_bert(batch):
    return bert_tokenizer(batch["text"], truncation=True)

train_ds_tok = train_ds.map(tokenize_bert, batched=True)
valid_ds_tok = valid_ds.map(tokenize_bert, batched=True)

bert_model = AutoModelForSequenceClassification.from_pretrained(bert_model_name, num_labels=2)
data_collator = DataCollatorWithPadding(tokenizer=bert_tokenizer)
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=preds, references=labels)

bert_output_dir = str(ARTIFACTS_DIR / "bert_classifier")

bert_args = TrainingArguments(
    output_dir=bert_output_dir,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=5,
    report_to="none",
)

bert_trainer = Trainer(
    model=bert_model,
    args=bert_args,
    train_dataset=train_ds_tok,
    eval_dataset=valid_ds_tok,
    processing_class=bert_tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

bert_trainer.train()
bert_metrics = bert_trainer.evaluate()
print(bert_metrics)

bert_trainer.save_model(bert_output_dir)
bert_tokenizer.save_pretrained(bert_output_dir)
print("Saved:", bert_output_dir)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/285 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/12 [00:00<?, ? examples/s]

Map:   0%|          | 0/6 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'eval_loss': 0.7111892104148865, 'eval_accuracy': 0.5, 'eval_runtime': 0.0254, 'eval_samples_per_second': 236.352, 'eval_steps_per_second': 78.784, 'epoch': 1.0}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved: /content/pyspark_deep_learning_demo/artifacts/bert_classifier


In [10]:
from transformers import pipeline

bert_pipe = pipeline(
    "text-classification",
    model=bert_output_dir,
    tokenizer=bert_output_dir,
    device=0 if torch.cuda.is_available() else -1
)

bert_examples = [
    "excellent product and wonderful support",
    "the app is awful and full of bugs",
    "it works but has a few issues"
]

bert_pipe(bert_examples)

Loading weights:   0%|          | 0/41 [00:00<?, ?it/s]

[{'label': 'LABEL_1', 'score': 0.534195601940155},
 {'label': 'LABEL_1', 'score': 0.5210909843444824},
 {'label': 'LABEL_1', 'score': 0.5098271369934082}]

In [11]:
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForCausalLM

llm_model_name = "sshleifer/tiny-gpt2"

def format_row(text, label):
    target = "positive" if int(label) == 1 else "negative"
    return f"Classify the sentiment of the following review.\n\nReview: {text}\nSentiment: {target}"

llm_train_pd = train_pd.copy()
llm_valid_pd = valid_pd.copy()

llm_train_pd["text_for_lm"] = [format_row(t, y) for t, y in zip(llm_train_pd["text"], llm_train_pd["label"])]
llm_valid_pd["text_for_lm"] = [format_row(t, y) for t, y in zip(llm_valid_pd["text"], llm_valid_pd["label"])]

llm_train_ds = Dataset.from_pandas(llm_train_pd[["text_for_lm"]], preserve_index=False)
llm_valid_ds = Dataset.from_pandas(llm_valid_pd[["text_for_lm"]], preserve_index=False)

llm_tokenizer = AutoTokenizer.from_pretrained(llm_model_name)
if llm_tokenizer.pad_token is None:
    llm_tokenizer.pad_token = llm_tokenizer.eos_token

def tokenize_llm(batch):
    out = llm_tokenizer(
        batch["text_for_lm"],
        truncation=True,
        padding="max_length",
        max_length=128,
    )
    out["labels"] = out["input_ids"].copy()
    return out

llm_train_tok = llm_train_ds.map(tokenize_llm, batched=True, remove_columns=["text_for_lm"])
llm_valid_tok = llm_valid_ds.map(tokenize_llm, batched=True, remove_columns=["text_for_lm"])

llm_base = AutoModelForCausalLM.from_pretrained(llm_model_name)
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["c_attn"],
)
llm_model = get_peft_model(llm_base, lora_config)
llm_model.print_trainable_parameters()

llm_output_dir = str(ARTIFACTS_DIR / "llm_lora")

llm_args = TrainingArguments(
    output_dir=llm_output_dir,
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    learning_rate=2e-4,
    logging_steps=5,
    report_to="none",
    fp16=False,
)

llm_trainer = Trainer(
    model=llm_model,
    args=llm_args,
    train_dataset=llm_train_tok,
    eval_dataset=llm_valid_tok,
)

llm_trainer.train()
llm_trainer.save_model(llm_output_dir)
llm_tokenizer.save_pretrained(llm_output_dir)
print("Saved:", llm_output_dir)

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:2285: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


trainable params: 128 || all params: 203,356 || trainable%: 0.0629


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,10.728961,10.727223


Saved: /content/pyspark_deep_learning_demo/artifacts/llm_lora


In [12]:
from transformers import AutoModelForCausalLM, AutoTokenizer

gen_tokenizer = AutoTokenizer.from_pretrained(llm_output_dir)
if gen_tokenizer.pad_token is None:
    gen_tokenizer.pad_token = gen_tokenizer.eos_token

gen_model = AutoModelForCausalLM.from_pretrained(llm_output_dir)

prompt = "Classify the sentiment of the following review.\n\nReview: excellent quality and fast shipping\nSentiment:"
inputs = gen_tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    output_ids = gen_model.generate(
        **inputs,
        max_new_tokens=12,
        do_sample=False
    )

print(gen_tokenizer.decode(output_ids[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/29 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: sshleifer/tiny-gpt2
Key                                   | Status     |  | 
--------------------------------------+------------+--+-
transformer.h.{0, 1}.attn.masked_bias | UNEXPECTED |  | 
transformer.h.{0, 1}.attn.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights: 0it [00:00, ?it/s]

GPT2LMHeadModel LOAD REPORT from: /content/pyspark_deep_learning_demo/artifacts/llm_lora
Key                                                                     | Status     | 
------------------------------------------------------------------------+------------+-
base_model.model.transformer.h.{0, 1}.attn.c_attn.lora_A.default.weight | UNEXPECTED | 
base_model.model.transformer.h.{0, 1}.attn.c_attn.lora_B.default.weight | UNEXPECTED | 
transformer.h.{0, 1}.attn.c_attn.lora_B.default.weight                  | MISSING    | 
transformer.h.{0, 1}.attn.c_attn.lora_A.default.weight                  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Classify the sentiment of the following review.

Review: excellent quality and fast shipping
Sentiment: factors factors factors factors factors factors factors factors factors factors factors factors


## Final notes

- **PySpark** is used for ETL and Parquet generation.
- **LSTM / CNN / BERT / LoRA** training happens outside Spark, which is the common modern pattern.
- For real projects, replace the sample CSV with your own data and increase epochs.
- For better transformer results, use a GPU runtime in Colab.

In [18]:
!zip -r pyspark_demo.zip /content/pyspark_deep_learning_demo

  adding: content/pyspark_deep_learning_demo/ (stored 0%)
  adding: content/pyspark_deep_learning_demo/artifacts/ (stored 0%)
  adding: content/pyspark_deep_learning_demo/artifacts/llm_lora/ (stored 0%)
  adding: content/pyspark_deep_learning_demo/artifacts/llm_lora/adapter_config.json (deflated 57%)
  adding: content/pyspark_deep_learning_demo/artifacts/llm_lora/adapter_model.safetensors (deflated 29%)
  adding: content/pyspark_deep_learning_demo/artifacts/llm_lora/README.md (deflated 65%)
  adding: content/pyspark_deep_learning_demo/artifacts/llm_lora/tokenizer_config.json (deflated 48%)
  adding: content/pyspark_deep_learning_demo/artifacts/llm_lora/checkpoint-6/ (stored 0%)
  adding: content/pyspark_deep_learning_demo/artifacts/llm_lora/checkpoint-6/adapter_config.json (deflated 57%)
  adding: content/pyspark_deep_learning_demo/artifacts/llm_lora/checkpoint-6/adapter_model.safetensors (deflated 29%)
  adding: content/pyspark_deep_learning_demo/artifacts/llm_lora/checkpoint-6/optimi

In [19]:
from google.colab import files
files.download("pyspark_demo.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>